In [ ]:
!pip -q install geemap earthengine-api

import ee
import geemap

ee.Authenticate()
ee.Initialize(project="remotesensinggi")

# =========================
# 1) USER SETTINGS
# =========================
COUNTRY_LABEL = "India"
DRIVE_FOLDER  = "CA_50"
SCALE         = 30

SEASON_START_MD = "-12-01"
SEASON_END_MD   = "-04-01"

THRESHOLDS = [0.3, 0.4, 0.5]
MIN_MONTHS_ABOVE = 2

earlyYears = [2000, 2001, 2002, 2003]
lateYears  = [2016, 2017, 2018, 2019]

# CA unique ID field 
ID_FIELD    = "Feature_ID"
OUT_ID_NAME = "ca_id"

# Export batching (rows per CSV)
BATCH_SIZE = 2

# reduceRegions tuning
TILE_SCALE = 16

LIMIT_PARTS_PER_RUN = False
START_PART = 0
MAX_PARTS_PER_RUN = 2

# ---- OPTIONAL DEBUG ----
DEBUG = False
DEBUG_CA_IDS = [6923, 7039]
DEBUG_YEAR   = 2018
DEBUG_THR    = 0.3
DEBUG_SCALE  = 120

# =========================
# 2) LOAD ASSETS
# =========================
# --- Command areas ---
ca_raw = ee.FeatureCollection("projects/remotesensinggi/assets/CA_50_India")
ca = ca_raw.map(lambda f: ee.Feature(f.geometry()).set(OUT_ID_NAME, f.get(ID_FIELD)))

# --- Cropland masks ---
crop2003 = ee.ImageCollection.fromImages([
    ee.Image("projects/remotesensinggi/assets/Cropland_NE_2003"),
    ee.Image("projects/remotesensinggi/assets/Cropland_NW_2003"),
    ee.Image("projects/remotesensinggi/assets/Cropland_SE_2003"),
    ee.Image("projects/remotesensinggi/assets/Cropland_SW_2003")
]).mosaic().gt(0).selfMask()

crop2019 = ee.ImageCollection.fromImages([
    ee.Image("projects/ee-endalkmamlove/assets/NE"),
    ee.Image("projects/ee-endalkmamlove/assets/NW"),
    ee.Image("projects/ee-endalkmamlove/assets/SE"),
    ee.Image("projects/ee-endalkmamlove/assets/SW")
]).mosaic().gt(0).selfMask()

# raster mask for ALL CA 
caMask = ee.Image(0).byte().paint(ca, 1).selfMask()

# geometry used to filter Landsat collections
region_geom = ca.geometry().bounds()

print("CA count:", ca.size().getInfo())

# =========================
# 3) CLOUD MASK + NDVI
# =========================
def cloud_mask_qa_pixel(img):
    qa = img.select("QA_PIXEL")
    mask = (qa.bitwiseAnd(1 << 0).eq(0)
            .And(qa.bitwiseAnd(1 << 1).eq(0))
            .And(qa.bitwiseAnd(1 << 3).eq(0))
            .And(qa.bitwiseAnd(1 << 4).eq(0)))
    return img.updateMask(mask)

def ndvi_l5_l7(img):
    img = cloud_mask_qa_pixel(img)
    red = img.select("SR_B3").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    return (nir.subtract(red).divide(nir.add(red))
            .rename("NDVI")
            .copyProperties(img, ["system:time_start"]))

def ndvi_l8_l9(img):
    img = cloud_mask_qa_pixel(img)
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)
    return (nir.subtract(red).divide(nir.add(red))
            .rename("NDVI")
            .copyProperties(img, ["system:time_start"]))

def get_ndvi_early(start, end, geom):
    l5 = (ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
          .filterDate(start, end).filterBounds(geom).map(ndvi_l5_l7))
    l7 = (ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
          .filterDate(start, end).filterBounds(geom).map(ndvi_l5_l7))
    return l5.merge(l7)

def get_ndvi_late(start, end, geom):
    l8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
          .filterDate(start, end).filterBounds(geom).map(ndvi_l8_l9))
    return l8

# =========================
# 4) SEASON + IRRIGATION LOGIC (correct masking)
# =========================
def season_dates_for_year(y: int):
    return ee.Date(f"{y}{SEASON_START_MD}"), ee.Date(f"{y+1}{SEASON_END_MD}")

def monthly_ndvi(ndvi_col, m_start, m_end, analysis_mask):
    return (ndvi_col.filterDate(m_start, m_end)
            .median()
            .rename("NDVI")
            .toFloat()
            .updateMask(analysis_mask))

def yearly_irrigation(ndvi_col, season_start, season_end, analysis_mask, thr: float):
    n_months = season_end.difference(season_start, "month").floor()
    offsets = ee.List.sequence(0, n_months.subtract(1))

    def per_month(m):
        m = ee.Number(m)
        m_start = season_start.advance(m, "month")
        m_end   = m_start.advance(1, "month")
        ndvi = monthly_ndvi(ndvi_col, m_start, m_end, analysis_mask)

        # Missing NDVI => not exceed (0)
        exceed = (ndvi.unmask(-9999)
                  .gte(thr)
                  .updateMask(analysis_mask)
                  .unmask(0)
                  .rename("exceed")
                  .uint8())
        return exceed

    exceed_count = ee.ImageCollection(offsets.map(per_month)).sum().rename("count").uint8()

    irrig = (exceed_count.gte(MIN_MONTHS_ABOVE)
             .updateMask(analysis_mask)
             .unmask(0)
             .rename("Irrigated")
             .uint8())
    return irrig

def final_period_irrigation(years, crop_mask, ndvi_getter):
    analysis_mask = caMask.updateMask(crop_mask)  # CA ∩ cropland
    final_by_thr = {t: ee.Image(0).uint8() for t in THRESHOLDS}

    for y in years:
        s, e = season_dates_for_year(y)
        ndvi_col = ndvi_getter(s, e, region_geom)
        for t in THRESHOLDS:
            irrig_y = yearly_irrigation(ndvi_col, s, e, analysis_mask, t)
            final_by_thr[t] = final_by_thr[t].max(irrig_y)

    return final_by_thr

early_final = final_period_irrigation(earlyYears, crop2003, get_ndvi_early)
late_final  = final_period_irrigation(lateYears,  crop2019, get_ndvi_late)

# =========================
# 5) OPTIONAL DEBUG (only a few CA IDs)
# =========================
def debug_ndvi_stats_for_ca(ca_id_value, year, crop_mask, ndvi_getter):
    feat = ca.filter(ee.Filter.eq(OUT_ID_NAME, ca_id_value)).first()
    geom = feat.geometry()
    s, e = season_dates_for_year(year)

    # local analysis mask: (this CA) ∩ cropland
    ca_raster = ee.Image(1).clip(geom).selfMask()
    analysis_mask_local = ca_raster.updateMask(crop_mask)

    ndvi_col = ndvi_getter(s, e, geom)

    seasonal_ndvi = (ndvi_col.median()
                     .rename("NDVI").toFloat()
                     .updateMask(analysis_mask_local))

    stats = seasonal_ndvi.reduceRegion(
        reducer=ee.Reducer.percentile([5, 50, 95]).combine(ee.Reducer.count(), "", True),
        geometry=geom,
        scale=DEBUG_SCALE,
        bestEffort=True,
        tileScale=16,
        maxPixels=1e13
    )

    exceed = seasonal_ndvi.unmask(-9999).gte(DEBUG_THR).rename("exceed").uint8()
    exceed_stats = exceed.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.count(), "", True),
        geometry=geom,
        scale=DEBUG_SCALE,
        bestEffort=True,
        tileScale=16,
        maxPixels=1e13
    )

    print(f"\n[DEBUG] CA={ca_id_value} year={year} season={s.format('YYYY-MM-dd').getInfo()} to {e.format('YYYY-MM-dd').getInfo()}")
    print(" NDVI p5/p50/p95 + count:", stats.getInfo())
    print(f" Exceed fraction thr={DEBUG_THR} + count:", exceed_stats.getInfo())

if DEBUG:
    print("\nRunning DEBUG on selected CA IDs...")
    for ca_id in DEBUG_CA_IDS:
        # choose which crop/ndvi getter makes sense for the debug year
        if DEBUG_YEAR <= 2013:
            debug_ndvi_stats_for_ca(ca_id, DEBUG_YEAR, crop2003, get_ndvi_early)
        else:
            debug_ndvi_stats_for_ca(ca_id, DEBUG_YEAR, crop2019, get_ndvi_late)

# =========================
# 6) PER-CA STATS + EXPORT 
# =========================
pixel_area = ee.Image.pixelArea()

KEEP_COLS = [
    OUT_ID_NAME, "country", "period", "threshold",
    "cmd_area_m2", "cropland_m2", "irrig_m2",
    "frac_cropland", "frac_irrig_in_cropland"
]

def per_ca_stats_fc(ca_subset, period_label, crop_mask, irrig_img, thr):
    crop_area = pixel_area.updateMask(caMask).updateMask(crop_mask).rename("cropland_m2")
    irrig_area = (pixel_area
                  .updateMask(caMask)
                  .updateMask(crop_mask)
                  .updateMask(irrig_img.eq(1))
                  .rename("irrig_m2"))

    img = ee.Image.cat([crop_area, irrig_area]).toFloat()

    reduced = img.reduceRegions(
        collection=ca_subset,
        reducer=ee.Reducer.sum(),
        scale=SCALE,
        tileScale=TILE_SCALE
    )

    def add_fields(f):
        cmd_area = ee.Number(f.geometry().area())
        crop_m2  = ee.Number(ee.Algorithms.If(f.get("cropland_m2"), f.get("cropland_m2"), 0))
        irrig_m2 = ee.Number(ee.Algorithms.If(f.get("irrig_m2"),   f.get("irrig_m2"),   0))

        frac_crop = ee.Algorithms.If(cmd_area.gt(0), crop_m2.divide(cmd_area), None)
        frac_irrig_in_crop = ee.Algorithms.If(crop_m2.gt(0), irrig_m2.divide(crop_m2), None)

        f2 = f.set({
            "country": COUNTRY_LABEL,
            "period": period_label,
            "threshold": thr,
            "cmd_area_m2": cmd_area,
            "frac_cropland": frac_crop,
            "frac_irrig_in_cropland": frac_irrig_in_crop
        })

        return ee.Feature(None, f2.toDictionary(KEEP_COLS))  # no geometry in export

    return reduced.map(add_fields)

def export_csv(fc, description):
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=description,
        folder=DRIVE_FOLDER,
        fileNamePrefix=description,
        fileFormat="CSV"
    )
    task.start()
    print("Started CSV:", description)

def export_perca_batched(period_label, crop_mask, irrig_img, thr, base_name):
    n = ca.size().getInfo()
    ca_list = ca.toList(n)

    n_parts = (n + BATCH_SIZE - 1) // BATCH_SIZE
    if LIMIT_PARTS_PER_RUN:
        part_start = START_PART
        part_end = min(START_PART + MAX_PARTS_PER_RUN, n_parts)
    else:
        part_start = 0
        part_end = n_parts

    print(f"\n{base_name}: exporting parts {part_start}..{part_end-1} of {n_parts-1}")

    for part in range(part_start, part_end):
        start = part * BATCH_SIZE
        subset = ee.FeatureCollection(ca_list.slice(start, start + BATCH_SIZE))
        fc_part = per_ca_stats_fc(subset, period_label, crop_mask, irrig_img, thr)
        export_csv(fc_part, f"{base_name}_part{part:03d}")

for t in THRESHOLDS:
    tstr = str(t).replace(".", "p")

    export_perca_batched(
        "2000_2003", crop2003, early_final[t], t,
        f"{COUNTRY_LABEL}_PerCA_2000_2003_NDVIgt{tstr}_B{BATCH_SIZE}"
    )

    export_perca_batched(
        "2016_2019", crop2019, late_final[t], t,
        f"{COUNTRY_LABEL}_PerCA_2016_2019_NDVIgt{tstr}_B{BATCH_SIZE}"
    )

print("\nAll exports started. Check Earth Engine Tasks + Drive folder:", DRIVE_FOLDER)
print("If LIMIT_PARTS_PER_RUN=True, rerun with START_PART =", START_PART + MAX_PARTS_PER_RUN)


CA count: 259

India_PerCA_2000_2003_NDVIgt0p3_B2: exporting parts 0..129 of 129
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part000
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part001
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part002
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part003
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part004
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part005
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part006
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part007
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part008
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part009
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part010
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part011
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part012
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part013
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part014
Started CSV: India_PerCA_2000_2003_NDVIgt0p3_B2_part015
Started CSV: India_PerC

In [ ]:
# =========================
# 1. Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:


# =========================
# 2. Imports
# =========================
import os
import glob
import pandas as pd

# =========================
# 3. Paths
# =========================
INPUT_FOLDER = "/content/drive/My Drive/CA_10"
OUTPUT_FOLDER = os.path.join(INPUT_FOLDER, "Merged")
OUTPUT_FILE = "CA-10.csv"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================
# 4. Find CSV files
# =========================
csv_files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.csv")))

print(f"Found {len(csv_files)} CSV files")

if not csv_files:
    raise FileNotFoundError("No CSV files found in the directory")

# =========================
# 5. Read and Merge
# =========================
dfs = []

for file in csv_files:
    try:
        df = pd.read_csv(file)
        df["source_file"] = os.path.basename(file)  # optional
        dfs.append(df)
    except Exception as e:
        print(f"Skipping {file} due to error:\n{e}")

merged_df = pd.concat(dfs, ignore_index=True)

# =========================
# 6. Save merged CSV
# =========================
output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
merged_df.to_csv(output_path, index=False)

print("✅ Merge complete!")
print(f"Saved to: {output_path}")
print(f"Rows: {len(merged_df):,}")
print(f"Columns: {len(merged_df.columns):,}")


Found 25 CSV files
✅ Merge complete!
Saved to: /content/drive/My Drive/CA_10/Merged/merged_all.csv
Rows: 2,514
Columns: 13
